# 24 — Equal-budget SigLIP B0–B4 screening (GPU, DEV-only)

Notebook này chạy ngay sau notebook 23 để chọn arm đáng đầu tư OCR/Qwen/BGE.
Nó dùng đúng 5.301 candidate cho mỗi B1–B4, vector SigLIP đã tạo ở notebook 13,
BTC baseline và 71 DEV event đã khóa. Không chạy lại decode/SigLIP candidate.

Đây là screening ở tầng SigLIP, không phải official full-feature score. Notebook
không đọc held-out, không mutate BTC và chỉ publish research artifacts. Candidate
NPZ được tải song song có retry để tránh 846 lượt Modal tuần tự quá chậm.

Chọn GPU T4/L4/A10, bật Internet và secrets GitHub/Modal. Attach Dataset
`mfe-data-v2` giống notebook 20/22. Giữ nguyên ô 0 rồi Save Version một lần.
Thành công khi log cuối có `SIGLIP_B0_B4_SCREEN_COMPLETE` và `events=71`.

## 0 — Khóa screening

In [ ]:
from pathlib import Path

RUN_SCOPE = "SCREEN"
CONFIRM_SCREEN = "B0-B4-SIGLIP-DEV"
ARMS = ("B0", "B1", "B2", "B3", "B4")
RUNNERS = ("quang", "trong")
EXPECTED_NB23_COMPLETE_SHA256 = "6f9f53c35024bec53e719794d60b818cf6ef9361d57ae1428b573b4be8c1be9a"
EXPECTED_PARITY_FREEZE_SHA256 = "3b9993755bb19000e80a9c7bd9ccdde1e9a4648dd1950167d0759f9a14227180"
EXPECTED_SELECTION_FREEZE_SHA256 = "75296b93ab4428ae721776d90167cdf3bca6b262137fd93610d04082def5a630"
EXPECTED_NB18_REPORT_SHA256 = "b089835a679809cfb1d2a1f79580343a8ceb0fe34846452038101f0db3a59228"
EXPECTED_TARGETS_SHA256 = "6929818a00205179f1d96c394f2930f43f88e9d536edccaf169b0632b5ee716b"
EXPECTED_TARGETS = 219
EXPECTED_DEV_QUERIES = 63
EXPECTED_DEV_TARGETS = 71
EXPECTED_BTC_SIGLIP_ROWS = 176707
EXPECTED_CANDIDATES_PER_ARM = 5301
EXPECTED_FEATURE_FRAMES = 17399
DOWNLOAD_WORKERS = 8
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 20260902

CHECKPOINT = "google/siglip-so400m-patch14-384"
MODEL_REVISION = "9fdffc58afc957d1a03a25b10dba0329ab15c2a3"
SIGLIP_DIMENSION = 1152
TEXT_MAX_LENGTH = 64
TEXT_BATCH_SIZE = 32

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO = Path("/kaggle/working/multimodal-fusion-engine")
WORK = Path("/kaggle/working/btc-siglip-b0-b4-screen")
REMOTE, OUTPUT = WORK / "remote", WORK / "output"
for path in (WORK, REMOTE, OUTPUT):
    path.mkdir(parents=True, exist_ok=True)
if RUN_SCOPE != "SCREEN" or CONFIRM_SCREEN != "B0-B4-SIGLIP-DEV":
    raise RuntimeError("SigLIP screening lock is closed")

## 1 — Repo, GPU và dependencies

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
import zipfile
from collections import Counter, defaultdict


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:4])}\n{result.stderr[-1800:]}")
    return result.stdout.strip()


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    github_token = secrets.get_secret("GITHUB_TOKEN")
    os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
    os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
except Exception as error:
    raise RuntimeError("Thiếu hoặc chưa bật quyền Kaggle Secrets GitHub/Modal") from error

askpass = WORK / "git-askpass.sh"
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({"GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    if (REPO / ".git").is_dir():
        if run(["git", "status", "--porcelain"], cwd=REPO):
            raise RuntimeError("Checkout Kaggle có local changes; Restart Session rồi Run All")
        run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, env=git_env)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    else:
        run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=git_env)
finally:
    askpass.unlink(missing_ok=True)


def torch_gpu_probe() -> dict:
    value = run([
        sys.executable,
        "-c",
        "import json,torch; c=torch.cuda.get_device_capability(0); "
        "print(json.dumps({'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0),"
        "'capability':list(c),'arch_list':torch.cuda.get_arch_list()}))",
    ])
    return json.loads(value.splitlines()[-1])


probe_before = torch_gpu_probe()
required_arch = f"sm_{probe_before['capability'][0]}{probe_before['capability'][1]}"
if required_arch not in probe_before["arch_list"]:
    run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
        "torch==2.12.1", "torchvision==0.27.1",
        "--index-url", "https://download.pytorch.org/whl/cu126",
    ])
probe_after = torch_gpu_probe()
if required_arch not in probe_after["arch_list"]:
    raise RuntimeError(f"PyTorch không hỗ trợ GPU được cấp: {probe_after}")

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "notebooks/requirements-kaggle-btc-images.txt")])
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
import yaml
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

from mfe.contracts.ids import build_keyframe_id, parse_keyframe_id
from mfe.core.config import load_config
from mfe.offline.augmentation.io import (
    canonical_hash,
    read_jsonl,
    sha256_file,
    write_json_atomic,
    write_jsonl_atomic,
)
from mfe.offline.augmentation.o1s_aggregation import (
    O1SEventTrace,
    adversarial_counterexample_summary,
    classify_siglip_failure_stage,
    cosine_scores,
    deterministic_best,
    deterministic_injected_rank,
    interval_candidate_indices,
    normalize_feature_rows,
    o1s_math_call_site_ledger,
    validate_o1s_worker_manifest,
)
from mfe.offline.augmentation.oracle_artifacts import OracleArtifactTarget
from mfe.offline.augmentation.representative_oracle import (
    RepresentativeResolvedSample,
    build_representative_scan_plan,
    partition_representative_scan,
    representative_scan_plan_hash,
    validate_scan_completion_marker,
)
from mfe.offline.bundle import BundleLayout, freeze_environment
from mfe.offline.source import load_catalogue
from mfe.offline.transport import ModalVolumeTransport
from mfe.online.translate import build_translator

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 24 cần GPU: bật Accelerator GPU rồi Run All lại")

repo_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO)
producer_config_path = REPO / "config/keyframe_augmentation_kaggle.yaml"
producer_config = yaml.safe_load(producer_config_path.read_text(encoding="utf-8"))
btc = ModalVolumeTransport(producer_config["btc"]["volume"], create_if_missing=False, version=2)
research = ModalVolumeTransport(producer_config["output"]["volume"], create_if_missing=False, version=2)
btc_layout = BundleLayout(producer_config["btc"]["bundle"])
upstream_root = producer_config["output"]["root"]
selection_root = f"{upstream_root}/gate1-selection-v1"
siglip_root = f"{upstream_root}/gate1-features-siglip-v1"
nb18_root = f"{upstream_root}/gate1-oracle-eval-quang-v1"
nb23_root = f"{upstream_root}/gate1-feature-parity-worklist-v1"
output_root = f"{upstream_root}/gate1-siglip-b0-b4-screen-v1"
print({"repo_commit": repo_commit, "gpu": torch.cuda.get_device_name(0), "scope": RUN_SCOPE})


def load_npz_vectors(path: Path) -> tuple[list[str], np.ndarray]:
    with np.load(path, allow_pickle=False) as artifact:
        if set(artifact.files) != {"ids", "vectors"}:
            raise RuntimeError(f"Unexpected vector NPZ members: {artifact.files}")
        ids = [str(value) for value in artifact["ids"].tolist()]
        vectors = np.asarray(artifact["vectors"], dtype=np.float32)
    return ids, vectors

## 2 — Khóa notebook 18, 23 và candidate manifests

In [ ]:
from datetime import datetime, timezone

nb18_path = REMOTE / "oracle-eval-report.json"
targets_path = REMOTE / "oracle-targets-resolved.jsonl"
nb23_complete_path = REMOTE / "nb23-COMPLETE.json"
parity_freeze_path = REMOTE / "parity-freeze.json"
selection_freeze_path = REMOTE / "selection-freeze.json"
research.get(f"{nb18_root}/oracle-eval-report.json", nb18_path)
research.get(f"{nb18_root}/oracle-targets-resolved.jsonl", targets_path)
research.get(f"{nb23_root}/COMPLETE.json", nb23_complete_path)
research.get(f"{nb23_root}/parity-freeze.json", parity_freeze_path)
research.get(f"{selection_root}/selection-freeze.json", selection_freeze_path)
for path, expected in ((nb18_path, EXPECTED_NB18_REPORT_SHA256), (targets_path, EXPECTED_TARGETS_SHA256), (nb23_complete_path, EXPECTED_NB23_COMPLETE_SHA256), (parity_freeze_path, EXPECTED_PARITY_FREEZE_SHA256), (selection_freeze_path, EXPECTED_SELECTION_FREEZE_SHA256)):
    if sha256_file(path) != expected:
        raise RuntimeError(f"Pinned input SHA drift: {path.name}")

nb18 = json.loads(nb18_path.read_text(encoding="utf-8"))
nb23_complete = json.loads(nb23_complete_path.read_text(encoding="utf-8"))
parity_freeze = json.loads(parity_freeze_path.read_text(encoding="utf-8"))
selection_freeze = json.loads(selection_freeze_path.read_text(encoding="utf-8"))
for payload, field in ((nb23_complete, "manifest_payload_hash"), (parity_freeze, "freeze_payload_hash"), (selection_freeze, "freeze_payload_hash")):
    unsigned = dict(payload); claimed = unsigned.pop(field, None)
    if claimed != canonical_hash(unsigned):
        raise RuntimeError(f"Payload hash mismatch: {field}")
if nb23_complete.get("status") != "COMPLETE" or parity_freeze.get("status") != "READY_FOR_FEATURE_PARITY_PRODUCERS":
    raise RuntimeError("Notebook 23 lineage is not ready")
if parity_freeze.get("feature_frames") != EXPECTED_FEATURE_FRAMES:
    raise RuntimeError("Notebook 23 feature frame count drift")

targets = [OracleArtifactTarget.model_validate(row) for row in read_jsonl(targets_path)]
if len(targets) != EXPECTED_TARGETS:
    raise RuntimeError("Oracle target count drift")
CANDIDATES = {}
for arm in ARMS[1:]:
    name = f"candidate-{arm.lower()}.jsonl"
    local = REMOTE / name
    research.get(f"{selection_root}/{name}", local)
    if sha256_file(local) != selection_freeze["files"][name]:
        raise RuntimeError(f"{arm}: candidate manifest SHA drift")
    rows = read_jsonl(local)
    for row in rows:
        row["keyframe_id"] = build_keyframe_id(str(row["video_id"]), int(row["frame_idx"]))
    if len(rows) != EXPECTED_CANDIDATES_PER_ARM or len({row["keyframe_id"] for row in rows}) != EXPECTED_CANDIDATES_PER_ARM:
        raise RuntimeError(f"{arm}: candidate identity/count drift")
    CANDIDATES[arm] = rows
print({"lineage": "PASS", "candidate_counts": {arm: len(rows) for arm, rows in CANDIDATES.items()}})

## 3 — Tải song song 17.399 vector candidate đã có

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

parity_worklist_path = REMOTE / "feature-parity-worklist.jsonl"
research.get(f"{nb23_root}/feature-parity-worklist.jsonl", parity_worklist_path)
if sha256_file(parity_worklist_path) != parity_freeze["files"]["feature-parity-worklist.jsonl"]:
    raise RuntimeError("Parity worklist SHA drift")
parity_rows = read_jsonl(parity_worklist_path)
if len(parity_rows) != EXPECTED_FEATURE_FRAMES:
    raise RuntimeError("Parity worklist count drift")

siglip_hash_by_video = {}
for runner in RUNNERS:
    local = REMOTE / f"runner-manifest-{runner}.json"
    research.get(f"{siglip_root}/runner-manifests/{runner}.json", local)
    manifest = json.loads(local.read_text(encoding="utf-8"))
    unsigned = dict(manifest); claimed = unsigned.pop("manifest_payload_hash", None)
    if claimed != canonical_hash(unsigned) or manifest.get("status") != "COMPLETE":
        raise RuntimeError(f"{runner}: notebook 13 manifest invalid")
    for row in manifest["verified"]:
        if row["video_id"] in siglip_hash_by_video:
            raise RuntimeError(f"Duplicate notebook 13 video ownership: {row['video_id']}")
        siglip_hash_by_video[row["video_id"]] = row["siglip_sha256"]

remote_by_video = {}
for row in parity_rows:
    remote_by_video.setdefault(row["video_id"], row["source_siglip"])
if set(remote_by_video) != set(siglip_hash_by_video):
    raise RuntimeError("Parity worklist/notebook13 video set mismatch")

def download_one(item: tuple[str, str]) -> tuple[str, Path]:
    video_id, remote_path = item
    local = REMOTE / "candidate-siglip" / f"{video_id}.npz"
    for attempt in range(3):
        try:
            research.get(remote_path, local)
            if sha256_file(local) != siglip_hash_by_video[video_id]:
                raise RuntimeError("SHA mismatch")
            return video_id, local
        except Exception:
            local.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(1 + attempt)
    raise AssertionError("unreachable")

downloaded = {}
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    futures = {pool.submit(download_one, item): item[0] for item in sorted(remote_by_video.items())}
    for ordinal, future in enumerate(as_completed(futures), 1):
        video_id, local = future.result()
        downloaded[video_id] = local
        if ordinal == 1 or ordinal % 100 == 0 or ordinal == len(futures):
            print({"candidate_siglip_downloaded": ordinal, "expected": len(futures)}, flush=True)

candidate_ids, candidate_vectors = [], []
for video_id in sorted(downloaded):
    ids, vectors = load_npz_vectors(downloaded[video_id])
    if vectors.shape != (len(ids), SIGLIP_DIMENSION) or not np.isfinite(vectors).all():
        raise RuntimeError(f"{video_id}: candidate vector shape/value invalid")
    candidate_ids.extend(ids); candidate_vectors.append(vectors)
if len(candidate_ids) != EXPECTED_FEATURE_FRAMES or len(set(candidate_ids)) != EXPECTED_FEATURE_FRAMES:
    raise RuntimeError("Combined candidate identity/count drift")
candidate_matrix = normalize_feature_rows(np.concatenate(candidate_vectors, axis=0))
candidate_position = {value: index for index, value in enumerate(candidate_ids)}
for arm, rows in CANDIDATES.items():
    missing = [row["keyframe_id"] for row in rows if row["keyframe_id"] not in candidate_position]
    if missing:
        raise RuntimeError(f"{arm}: missing vectors {missing[:3]}")
print({"candidate_vectors": len(candidate_ids), "videos": len(downloaded), "dimension": candidate_matrix.shape[1]})

## 4 — BTC global SigLIP và DEV queryset

In [ ]:
def resolve_base_siglip() -> tuple[Path, dict[str, str]]:
    input_root = Path("/kaggle/input")
    member = "stores/flat/data/vectors/siglip_vectors.npz"
    expanded = sorted(input_root.rglob(member))
    if len(expanded) == 1:
        return expanded[0], {"mode": "expanded_kaggle_dataset", "sha256": sha256_file(expanded[0])}
    if len(expanded) > 1:
        raise RuntimeError(f"Có nhiều hơn một base SigLIP trong Kaggle Input: {expanded}")

    archives = sorted(input_root.rglob("mfe-data-v2.zip"))
    if len(archives) == 1:
        extracted = WORK / "base-input" / member
        with zipfile.ZipFile(archives[0]) as zipped:
            if member not in set(zipped.namelist()):
                raise RuntimeError(f"{archives[0]} thiếu {member}")
            expected_size = zipped.getinfo(member).file_size
            if not extracted.is_file() or extracted.stat().st_size != expected_size:
                extracted.parent.mkdir(parents=True, exist_ok=True)
                partial = extracted.with_suffix(extracted.suffix + ".part")
                partial.unlink(missing_ok=True)
                try:
                    with zipped.open(member) as source, partial.open("wb") as destination:
                        shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)
                    if partial.stat().st_size != expected_size:
                        raise RuntimeError("Extracted base SigLIP size mismatch")
                    partial.replace(extracted)
                finally:
                    partial.unlink(missing_ok=True)
        return extracted, {"mode": "mfe_data_v2_zip", "sha256": sha256_file(extracted)}
    if len(archives) > 1:
        raise RuntimeError(f"Có nhiều hơn một mfe-data-v2.zip trong Kaggle Input: {archives}")

    remote = btc_layout.flat("siglip_vectors")
    if btc.exists(remote):
        downloaded = WORK / "btc-siglip-vectors.npz"
        btc.get(remote, downloaded)
        return downloaded, {"mode": "keyframes_btc_flat", "sha256": sha256_file(downloaded)}
    raise RuntimeError(
        "Thiếu BTC global SigLIP. Hãy Add Input đúng Dataset mfe-data-v2 đã dùng ở "
        "notebook 18 (có stores/flat/data/vectors/siglip_vectors.npz hoặc mfe-data-v2.zip), "
        "rồi Save Version lại. Không cần chạy lại notebook 19."
    )


base_siglip_path, base_siglip_source = resolve_base_siglip()
expected_base_siglip_hash = nb18["input_hashes"]["base_siglip"]
if sha256_file(base_siglip_path) != expected_base_siglip_hash:
    raise RuntimeError("BTC global SigLIP hash differs from notebook 18 baseline")
btc_ids_raw, btc_vectors_raw = load_npz_vectors(base_siglip_path)
btc_ids = [str(value) for value in btc_ids_raw]
btc_matrix = normalize_feature_rows(np.asarray(btc_vectors_raw, dtype=np.float32))
if len(btc_ids) != EXPECTED_BTC_SIGLIP_ROWS or len(set(btc_ids)) != EXPECTED_BTC_SIGLIP_ROWS:
    raise RuntimeError("BTC global SigLIP identity count drift")
if btc_matrix.shape != (EXPECTED_BTC_SIGLIP_ROWS, SIGLIP_DIMENSION):
    raise RuntimeError(f"BTC global SigLIP shape drift: {btc_matrix.shape}")

queryset_path = REPO / "eval/queryset_dev.jsonl"
if sha256_file(queryset_path) != nb18["input_hashes"]["dev_queryset"]:
    raise RuntimeError("DEV queryset hash differs from notebook 18")
queries = read_jsonl(queryset_path)
query_by_id = {str(row["query_id"]): row for row in queries}
if len(query_by_id) != EXPECTED_DEV_QUERIES:
    raise RuntimeError(f"Expected {EXPECTED_DEV_QUERIES} DEV queries, got {len(query_by_id)}")
dev_targets = sorted(
    (target for target in targets if target.query_id in query_by_id),
    key=lambda target: (target.query_id, target.event_index),
)
if len(dev_targets) != EXPECTED_DEV_TARGETS:
    raise RuntimeError(f"Expected {EXPECTED_DEV_TARGETS} DEV target events, got {len(dev_targets)}")
print({"btc_rows": len(btc_ids), "dev_queries": len(query_by_id), "dev_target_events": len(dev_targets), "btc_mode": "READ_ONLY", "base_siglip_source": base_siglip_source})

## 5 — Query text và SigLIP text tower

In [ ]:
def event_query_text(query: dict, event_index: int) -> str:
    if query["type"] == "TRAKE":
        texts = query.get("event_texts") or []
        if not 0 <= event_index < len(texts):
            raise RuntimeError(f"{query['query_id']}: missing TRAKE event text {event_index}")
        return str(texts[event_index])
    if query["type"] == "QA":
        return str(query.get("retrieval_text") or query["text"])
    return str(query["text"])


translation_root = WORK / "translation-root"
translation_cache_input = REPO / "eval/translations.json"
translation_cache_runtime = translation_root / "eval/translations.json"
translation_cache_runtime.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(translation_cache_input, translation_cache_runtime)
translation_cache_input_hash = sha256_file(translation_cache_input)
config = load_config(REPO / "config/btc_b1.yaml", repo_root=REPO)
translator = build_translator(config, repo_root=translation_root)
if translator is None:
    raise RuntimeError("Production translation contract is disabled")

translation_rows = []
event_text_en = {}
for target in dev_targets:
    key = (target.query_id, target.event_index)
    query = query_by_id[target.query_id]
    text_vi = event_query_text(query, target.event_index)
    translation = translator.translate(text_vi, manual=query.get("text_en"))
    if translation is None:
        raise RuntimeError(f"{target.query_id}/{target.event_index}: no English SigLIP query")
    event_text_en[key] = translation.text_en
    translation_rows.append({
        "query_id": target.query_id,
        "event_index": target.event_index,
        "text_vi": text_vi,
        "text_en": translation.text_en,
        "tier": translation.tier,
        "model": translation.model,
    })
translator.save_cache()
translation_cache_runtime_hash = sha256_file(translation_cache_runtime)
translations_path = write_jsonl_atomic(OUTPUT / "translations-used.jsonl", translation_rows)
translation_usage = dict(translator.usage)
del translator
gc.collect()

snapshot_dir = Path(snapshot_download(CHECKPOINT, revision=MODEL_REVISION))
model_file_hashes = {
    str(path.relative_to(snapshot_dir)): sha256_file(path)
    for path in sorted(snapshot_dir.rglob("*"))
    if path.is_file()
}
model_snapshot_hash = canonical_hash(model_file_hashes)
tokenizer = AutoTokenizer.from_pretrained(snapshot_dir, local_files_only=True)
model = AutoModel.from_pretrained(snapshot_dir, local_files_only=True).eval().cuda()

ordered_event_keys = [(target.query_id, target.event_index) for target in dev_targets]
query_vectors = []
for start in range(0, len(ordered_event_keys), TEXT_BATCH_SIZE):
    keys = ordered_event_keys[start : start + TEXT_BATCH_SIZE]
    batch = tokenizer(
        [event_text_en[key] for key in keys],
        padding="max_length",
        max_length=TEXT_MAX_LENGTH,
        truncation=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        output = model.get_text_features(**batch)
        pooled = getattr(output, "pooler_output", None)
        features = pooled if pooled is not None else output
    query_vectors.append(features.float().cpu().numpy())
query_matrix = normalize_feature_rows(np.concatenate(query_vectors, axis=0))
if query_matrix.shape != (EXPECTED_DEV_TARGETS, SIGLIP_DIMENSION):
    raise RuntimeError(f"SigLIP query matrix shape drift: {query_matrix.shape}")
del model, tokenizer, query_vectors, batch, output, pooled, features
gc.collect()
torch.cuda.empty_cache()
print({"translated_events": len(translation_rows), "translation_usage": translation_usage, "model_snapshot_hash": model_snapshot_hash})

## 6 — Equal-budget rank metrics và paired bootstrap

In [ ]:
with torch.inference_mode():
    query_tensor = torch.from_numpy(query_matrix).to("cuda")
    btc_scores = (query_tensor @ torch.from_numpy(btc_matrix).to("cuda").T).float().cpu().numpy()
    candidate_scores = (query_tensor @ torch.from_numpy(candidate_matrix).to("cuda").T).float().cpu().numpy()
del query_tensor
torch.cuda.empty_cache()

def best_rank(ids: list[str], scores: np.ndarray, relevant_positions: list[int]) -> tuple[int | None, str | None, float | None]:
    if not relevant_positions:
        return None, None, None
    relevant_ids = [ids[index] for index in relevant_positions]
    relevant_scores = scores[relevant_positions]
    best_id, best_score, _ = deterministic_best(relevant_ids, relevant_scores)
    return deterministic_injected_rank(ids, scores, candidate_id=best_id, candidate_score=best_score), best_id, best_score

arm_ids = {"B0": btc_ids}
arm_positions = {}
for arm in ARMS[1:]:
    ids = [row["keyframe_id"] for row in CANDIDATES[arm]]
    arm_ids[arm] = btc_ids + ids
    arm_positions[arm] = np.asarray([candidate_position[value] for value in ids], dtype=np.int64)

EVENT_ROWS = []
metric_by_arm = defaultdict(list)
for event_position, target in enumerate(dev_targets):
    btc_relevant = interval_candidate_indices(btc_ids, video_id=target.video_id, start_frame=target.frame_range_start, end_frame=target.frame_range_end).tolist()
    for arm in ARMS:
        if arm == "B0":
            scores = btc_scores[event_position]
            relevant = btc_relevant
        else:
            positions = arm_positions[arm]
            extra_ids = arm_ids[arm][len(btc_ids):]
            extra_scores = candidate_scores[event_position, positions]
            scores = np.concatenate((btc_scores[event_position], extra_scores))
            extra_relevant = [index for index, value in enumerate(extra_ids) if (lambda parsed: parsed[0] == target.video_id and target.frame_range_start <= parsed[1] <= target.frame_range_end)(parse_keyframe_id(value))]
            relevant = btc_relevant + [len(btc_ids) + index for index in extra_relevant]
        rank, keyframe_id, score = best_rank(arm_ids[arm], scores, relevant)
        rr = 0.0 if rank is None else 1.0 / rank
        metric_by_arm[arm].append(rr)
        EVENT_ROWS.append({"query_id": target.query_id, "event_index": target.event_index, "video_id": target.video_id, "arm": arm, "rank": rank, "reciprocal_rank": rr, "best_relevant_keyframe_id": keyframe_id, "best_relevant_score": score, "relevant_count": len(relevant)})
    if (event_position + 1) % 10 == 0 or event_position + 1 == len(dev_targets):
        print({"measured_events": event_position + 1, "expected": len(dev_targets)}, flush=True)

rng = np.random.default_rng(BOOTSTRAP_SEED)
SUMMARY = {}
b0 = np.asarray(metric_by_arm["B0"], dtype=np.float64)
for arm in ARMS:
    values = np.asarray(metric_by_arm[arm], dtype=np.float64)
    delta = values - b0
    if arm == "B0":
        low = high = 0.0
    else:
        draws = rng.integers(0, len(delta), size=(BOOTSTRAP_SAMPLES, len(delta)))
        boot = delta[draws].mean(axis=1)
        low, high = np.quantile(boot, [0.025, 0.975]).tolist()
    ranks = [row["rank"] for row in EVENT_ROWS if row["arm"] == arm]
    SUMMARY[arm] = {"event_mrr": float(values.mean()), "delta_mrr_vs_b0": float(delta.mean()), "paired_bootstrap_95ci": [float(low), float(high)], "events_better_than_b0": int(np.sum(delta > 0)), "events_worse_than_b0": int(np.sum(delta < 0)), "recall_at_1": float(np.mean([rank is not None and rank <= 1 for rank in ranks])), "recall_at_5": float(np.mean([rank is not None and rank <= 5 for rank in ranks])), "recall_at_10": float(np.mean([rank is not None and rank <= 10 for rank in ranks])), "recall_at_100": float(np.mean([rank is not None and rank <= 100 for rank in ranks]))}

winner = max(ARMS[1:], key=lambda arm: (SUMMARY[arm]["event_mrr"], arm))
winner_ci = SUMMARY[winner]["paired_bootstrap_95ci"]
decision = "SIGLIP_SCREEN_POSITIVE" if SUMMARY[winner]["delta_mrr_vs_b0"] > 0 and winner_ci[0] > 0 else "SIGLIP_SCREEN_INCONCLUSIVE"
print({"arm_summary": SUMMARY, "provisional_winner": winner, "decision": decision})

## 7 — Publish screening report

In [ ]:
events_path = write_jsonl_atomic(OUTPUT / "siglip-screen-events.jsonl", EVENT_ROWS)
report = {"schema_version": 1, "status": "SIGLIP_B0_B4_SCREEN_COMPLETE", "decision_authority": "siglip_only_arm_screen", "repo_commit": repo_commit, "events": len(dev_targets), "dev_queries": len(query_by_id), "equal_budget": EXPECTED_CANDIDATES_PER_ARM, "candidate_feature_frames": EXPECTED_FEATURE_FRAMES, "arms": SUMMARY, "provisional_winner": winner, "decision": decision, "bootstrap": {"paired": True, "samples": BOOTSTRAP_SAMPLES, "seed": BOOTSTRAP_SEED, "unit": "target_event"}, "input_hashes": {"nb23_complete": EXPECTED_NB23_COMPLETE_SHA256, "parity_freeze": EXPECTED_PARITY_FREEZE_SHA256, "selection_freeze": EXPECTED_SELECTION_FREEZE_SHA256, "nb18_report": EXPECTED_NB18_REPORT_SHA256, "oracle_targets": EXPECTED_TARGETS_SHA256}, "full_feature_score_status": "BLOCKED_PENDING_WINNER_FEATURE_PARITY", "heldout_accessed": False, "btc_mutated": False, "created_at_utc": datetime.now(timezone.utc).isoformat()}
report["report_payload_hash"] = canonical_hash(report)
report_path = write_json_atomic(OUTPUT / "siglip-screen-report.json", report)
uploads = [(events_path, f"{output_root}/{events_path.name}"), (report_path, f"{output_root}/{report_path.name}")]
research.put_many(uploads)
for local, remote in uploads:
    if research.sha256(remote) != sha256_file(local):
        raise RuntimeError(f"Remote output verify failed: {remote}")
complete = {"schema_version": 1, "status": "COMPLETE", "report_sha256": sha256_file(report_path), "events_sha256": sha256_file(events_path), "provisional_winner": winner, "decision": decision, "next": "Run OCR-classical and Qwen25 in parallel for the provisional winner; BGE-M3 follows Qwen25 per completed shard.", "heldout_accessed": False, "btc_mutated": False}
complete["manifest_payload_hash"] = canonical_hash(complete)
complete_path = write_json_atomic(OUTPUT / "COMPLETE.json", complete)
research.put(complete_path, f"{output_root}/COMPLETE.json")
if research.sha256(f"{output_root}/COMPLETE.json") != sha256_file(complete_path):
    raise RuntimeError("Remote COMPLETE verify failed")
print({"SIGLIP_B0_B4_SCREEN_COMPLETE": True, "events": len(dev_targets), "winner": winner, "decision": decision, "arm_summary": SUMMARY, "complete_sha256": sha256_file(complete_path)})

## 8 — Cách dùng kết quả

`provisional_winner` chỉ chọn arm để tiết kiệm feature extraction. Nếu bootstrap
CI thấp hơn vẫn không vượt 0, screening là chưa chắc chắn nhưng vẫn có thể chọn
arm MRR cao nhất để chạy full parity. Chỉ aggregate full-feature mới trả lời điểm
end-to-end có tăng thực sự hay không.